# IFC PATH ANALYSIS — SHORTEST PATHS, WORST CASE & AVERAGE SCENARIO

## 1. Import the needed classes

In [1]:
import sys #needed for vscode to recognize the topologicpy packagefrom topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Grid import Grid
from topologicpy.Color import Color
from topologicpy.Helper import Helper
from topologicpy.IFC import IFC

## 2. Check the TopologicPy version

In [2]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.35) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [3]:
renderer = "vscode"

## 4. Load the IFC file and extract space topologies
We extract `IfcSpace` elements as they represent occupiable rooms/areas that will form the nodes of our navigation graph.

In [4]:
# --- Set your IFC file path ---
ifc_file_path = r"C:\Users\Win11\Desktop\AIA2026\GraphLM\Updated Notebooks\Graph ML -- NOTEBOOKS\Supporting Files\NVW_DCR-LOD200_Arch.ifc"  

ifc_file = IFC.FileByPath(ifc_file_path)

# List all element types available in the file
element_types = IFC.ElementTypes(ifc_file, includeCounts=False, silent=False)
element_types = [e.lower() for e in element_types]
print("Available element types:")
for et in element_types:
    print(" ", et)

IFCFastTopology.Parse - Parsed 515000 entities in 4.738s.
Available element types:
  ifcapplication
  ifcarbitraryclosedprofiledef
  ifcarbitraryprofiledefwithvoids
  ifcaxis2placement2d
  ifcaxis2placement3d
  ifcbooleanclippingresult
  ifcboundingbox
  ifcbuilding
  ifcbuildingelementproxy
  ifcbuildingstorey
  ifccartesianpoint
  ifccartesiantransformationoperator3d
  ifcclassificationreference
  ifccolumn
  ifcconnectionsurfacegeometry
  ifcconversionbasedunit
  ifccurveboundedplane
  ifcdimensionalexponents
  ifcdirection
  ifcdoor
  ifcdoorstyle
  ifcelementquantity
  ifcextrudedareasolid
  ifcface
  ifcfaceouterbound
  ifcgeometricrepresentationcontext
  ifclocalplacement
  ifcmappeditem
  ifcmaterial
  ifcmateriallayer
  ifcmateriallayerset
  ifcmateriallayersetusage
  ifcmateriallist
  ifcmeasurewithunit
  ifcmember
  ifcopeningelement
  ifcopenshell
  ifcorganization
  ifcownerhistory
  ifcperson
  ifcpersonandorganization
  ifcplane
  ifcpolygonalboundedhalfspace
  ifcpolyli

In [5]:
# Extract IfcSpace topologies — these represent the navigable rooms/zones
# Adjust includeTypes as needed (e.g. add "ifczone" if your model uses zones)
topologies = IFC.TopologiesByPath(
    ifc_file_path,
    includeTypes=["ifcspace"],
    excludeTypes=[],
    dictionaryMode="all",
    clean=False,
    epsilon=0.0001,
    angTolerance=0.1,
    tolerance=0.0001,
    silent=False
)
print(f"Imported {len(topologies)} IfcSpace objects.")

IFCFastTopology.Parse - Parsed 515000 entities in 4.731s.
IFCFastTopology.TopologiesByEntities - Created 89 topologies; skipped 0 products in 3.635s.
Imported 89 IfcSpace objects.


## 5. Show the IFC spaces

In [6]:
Topology.Show(
    topologies,
    faceOpacity=0.3,
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 6. Build a 2D floor-plan shell from the IFC spaces
We extract the floor faces of each space to create a flat walkable shell, then slice it with a regular grid to produce the analysis mesh.

In [8]:
# Collect all faces from the space topologies and build a shell
# Source: S04-11B — Cluster.Faces + Shell.ByFaces
all_faces = Cluster.Faces(Cluster.ByTopologies(topologies))
floor_plan = Shell.ByFaces(all_faces)
print("Floor plan shell created:", floor_plan)

Topology.SelfMerge - Error: The input topology is not a valid topology. Returning None
Shell.ByFaces - Error: Could not create shell. Returning None.
Floor plan shell created: None


In [ ]:
Topology.Show(
    floor_plan,
    camera=[0, 0, 6],
    faceColor=[210, 210, 250],
    faceOpacity=1,
    edgeColor="white",
    edgeWidth=3,
    showVertices=False,
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 7. Create a grid overlay and slice the floor plan
This is step **1 — Create a sampled / waypoint graph**.
The grid spacing controls the resolution of your path analysis: smaller = more accurate, slower.

In [ ]:
# Compute bounding rectangle of the floor plan
b_r = Wire.BoundingRectangle(floor_plan)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")

# Grid spacing in model units (adjust to your IFC unit — typically metres or mm)
grid_spacing = 1.5  # e.g. 1.5 m spacing — increase to speed up, decrease for accuracy

uRange = list(range(0, int(width) + grid_spacing + 1, grid_spacing))
vRange = list(range(0, int(length) + grid_spacing + 1, grid_spacing))

# Create a grid of edges for slicing the floor plan
grid = Grid.EdgesByDistances(floor_plan, clip=True, uRange=uRange, vRange=vRange)

# Slice the floor plan shell with the grid
shell = Topology.Slice(floor_plan, grid)

# Assign a unique id to each face so we can track analysis results back to it
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    f = Topology.SetDictionary(f, d)

print(f"Shell has {len(faces)} faces after slicing.")

## 8. Build the navigation graph and the compiled routing graph
- `navigation_graph` connects faces that share topology (edges/vertices), giving us a traversal graph.
- `Graph.CompiledRoutingGraph` pre-processes the graph for fast shortest-path queries.

In [ ]:
# Navigation graph: faces connected through shared topologies (edge-to-edge adjacency)
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)

# Compiled routing graph: pre-processed for fast repeated shortest-path queries
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)

# Also keep the analysis graph (direct adjacency) for later visualisation
analysis_graph = Graph.ByTopology(shell)
g_verts = Graph.Vertices(analysis_graph)

print(f"Navigation graph: {len(Graph.Vertices(navigation_graph))} vertices, {len(Graph.Edges(navigation_graph))} edges.")

In [ ]:
Topology.Show(
    floor_plan, navigation_graph,
    camera=[0, 0, 6],
    faceColor=[210, 210, 250],
    faceOpacity=0.4,
    vertexSize=3,
    vertexColor="red",
    edgeColor="lightgrey",
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 9. Define source (starting) points and destination points
Steps **2** and **3** — list all starting points and all destination points.

You can define these explicitly with coordinates, or derive them from IFC elements such as doors, stairs, or specific named spaces.

In [ ]:
# --- Option A: Explicit coordinates ---
# Replace with your actual source and destination coordinates (x, y, z)
source_coords = [
    (xmin + 2, ymin + 2, 0),  # e.g. entrance / exit 1
    (xmin + 2, ymax - 2, 0),  # e.g. entrance / exit 2
]

destination_coords = [
    (xmax - 2, ymax - 2, 0),  # e.g. assembly point 1
    (xmax - 2, ymin + 2, 0),  # e.g. assembly point 2
    ((xmin + xmax) / 2, (ymin + ymax) / 2, 0),  # e.g. central hub
]

source_vertices = [Vertex.ByCoordinates(*c) for c in source_coords]
destination_vertices = [Vertex.ByCoordinates(*c) for c in destination_coords]

print(f"{len(source_vertices)} source points, {len(destination_vertices)} destination points.")

In [ ]:
# --- Option B: Derive from IFC doors (all doorway centroids become sources AND destinations) ---
# Uncomment and adapt if you want to route between all door openings automatically.

# door_topologies = IFC.TopologiesByPath(
#     ifc_file_path,
#     includeTypes=["ifcdoor"],
#     dictionaryMode="all",
#     silent=True
# )
# door_vertices = [Topology.Centroid(d) for d in door_topologies]
# source_vertices = door_vertices
# destination_vertices = door_vertices
# print(f"{len(door_vertices)} door centroids loaded as source and destination points.")

## 10. Compute shortest paths from every source to every destination
Step **4** — compute shortest paths. For each (source, destination) pair we:
- Find the shortest graph path.
- Optionally straighten it (step **5**).
- Measure its length (step **6**).

In [ ]:
results = []  # will hold dicts: {source, destination, path, straight_path, raw_length, straight_length}

for i, sv in enumerate(source_vertices):
    for j, dv in enumerate(destination_vertices):
        label = f"S{i+1}→D{j+1}"

        # Skip identical source-destination
        if Vertex.Distance(sv, dv) < 0.01:
            continue

        # --- Step 4: Compute shortest path ---
        t0 = time.time()
        path = Graph.ShortestPath(crg, vertexA=sv, vertexB=dv)
        t1 = time.time()

        if path is None:
            print(f"{label}: No path found (source or destination may be outside the floor plan).")
            continue

        raw_length = Wire.Length(path)

        # --- Step 5: Straighten the path (optional) ---
        straight_path = Wire.Straighten(path, host=floor_plan)
        straight_length = Wire.Length(straight_path) if straight_path else raw_length

        results.append({
            "label": label,
            "source_idx": i,
            "dest_idx": j,
            "path": path,
            "straight_path": straight_path,
            "raw_length": raw_length,
            "straight_length": straight_length,
            "duration_s": round(t1 - t0, 3)
        })

        print(f"{label} | raw: {round(raw_length,2)} m | straightened: {round(straight_length,2)} m | ({round(t1-t0,2)}s)")

## 11. Analyse results — worst case, best case, average
Step **7** — conduct analysis on worst case scenario, average path, etc.

In [ ]:
if results:
    lengths = [r["straight_length"] for r in results]

    worst  = max(results, key=lambda r: r["straight_length"])
    best   = min(results, key=lambda r: r["straight_length"])
    avg_length = sum(lengths) / len(lengths)

    print("=" * 50)
    print(f"Total paths computed : {len(results)}")
    print(f"Worst case  [{worst['label']}]: {round(worst['straight_length'], 2)} m")
    print(f"Best case   [{best['label']}]:  {round(best['straight_length'], 2)} m")
    print(f"Average path length  : {round(avg_length, 2)} m")
    print("=" * 50)
else:
    print("No paths were computed. Check that your source/destination points fall inside the floor plan.")

## 12. Colour-code all paths and show them
Red = raw shortest path graph walk. Blue = straightened path.

In [ ]:
all_path_edges = []

for r in results:
    # Raw path in red
    for edge in Topology.Edges(r["path"]):
        d = Dictionary.ByKeysValues(["width", "color"], [4, "red"])
        edge = Topology.SetDictionary(edge, d)
        all_path_edges.append(edge)
    # Straightened path in blue
    if r["straight_path"]:
        for edge in Topology.Edges(r["straight_path"]):
            d = Dictionary.ByKeysValues(["width", "color"], [4, "blue"])
            edge = Topology.SetDictionary(edge, d)
            all_path_edges.append(edge)

# Colour source vertices green, destination vertices yellow
for v in source_vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [20, "green"])
    v = Topology.SetDictionary(v, d)
for v in destination_vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [20, "yellow"])
    v = Topology.SetDictionary(v, d)

Topology.Show(
    floor_plan,
    *all_path_edges,
    *source_vertices,
    *destination_vertices,
    camera=[0, 0, 6],
    faceColor=[210, 210, 250],
    faceOpacity=0.3,
    edgeColorKey="color",
    edgeWidthKey="width",
    vertexSizeKey="size",
    vertexColorKey="color",
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)
print("Red = raw shortest path | Blue = straightened path | Green = sources | Yellow = destinations")

## 13. Highlight the worst-case path separately

In [ ]:
if results:
    worst_edges = []
    for edge in Topology.Edges(worst["path"]):
        d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
        edge = Topology.SetDictionary(edge, d)
        worst_edges.append(edge)
    if worst["straight_path"]:
        for edge in Topology.Edges(worst["straight_path"]):
            d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
            edge = Topology.SetDictionary(edge, d)
            worst_edges.append(edge)

    print(f"Worst case path: {worst['label']} — straightened length {round(worst['straight_length'],2)} m")

    Topology.Show(
        floor_plan,
        *worst_edges,
        camera=[0, 0, 6],
        faceColor=[210, 210, 250],
        faceOpacity=0.3,
        edgeColorKey="color",
        edgeWidthKey="width",
        backgroundColor="black",
        width=800,
        height=600,
        renderer=renderer
    )

## 14. Print a summary table of all paths

In [ ]:
print(f"{'Path':<12} {'Raw length (m)':>16} {'Straight length (m)':>20} {'Reduction (m)':>14}")
print("-" * 66)
for r in sorted(results, key=lambda x: x["straight_length"], reverse=True):
    reduction = round(r["raw_length"] - r["straight_length"], 2)
    print(f"{r['label']:<12} {round(r['raw_length'],2):>16} {round(r['straight_length'],2):>20} {reduction:>14}")

if results:
    print("-" * 66)
    print(f"{'AVERAGE':<12} {round(sum(r['raw_length'] for r in results)/len(results),2):>16} {round(avg_length,2):>20}")
    print(f"{'WORST':<12} {round(worst['raw_length'],2):>16} {round(worst['straight_length'],2):>20}")
    print(f"{'BEST':<12} {round(best['raw_length'],2):>16} {round(best['straight_length'],2):>20}")